<a href="https://www.kaggle.com/code/kbmhkhan/2-sheed-multiple-phase-version1?scriptVersionId=359940651" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# 01 — IMPORTS + EXPERIMENT CONFIG
# ============================================================
import os, gc, json, time, math, copy, random, shutil, warnings, traceback, re
from pathlib import Path
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torchvision
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from scipy import stats
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS     = ["mobilenet_v3_large", "resnet50", "densenet121", "efficientnet_b0"]
RUN_MODELS = MODELS
SEEDS      = [42, 123, 2024, 3407, 777]

CLASS_NAMES  = ["Cardboard", "Glass", "Metal", "Paper", "Plastic"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 4
WEIGHT_DECAY = 1e-4

P1_EPOCHS = 8
P1_LR     = 1e-3
P2_MAX_EPOCHS   = 30
P2_BACKBONE_LR  = 2e-5
P2_HEAD_LR      = 1e-4
P2_PATIENCE     = 7
SCHED_PATIENCE  = 2

USE_CLASS_WEIGHTS = True
USE_AMP   = True
ECE_BINS  = 15
AMP_ON    = USE_AMP and DEVICE.type == "cuda"

HEAD_PREFIX = {
    "resnet50":           ["fc"],
    "densenet121":        ["classifier"],
    "efficientnet_b0":    ["classifier"],
    "mobilenet_v3_large": ["classifier"],
}
UNFREEZE_PREFIX = {
    "resnet50":           ["layer4"],
    "densenet121":        ["features.denseblock4", "features.norm5"],
    "efficientnet_b0":    ["features.6", "features.7", "features.8"],
    "mobilenet_v3_large": ["features.13", "features.14", "features.15", "features.16"],
}

KAGGLE_INPUT = Path("/kaggle/input")
OUT_DIR   = Path("/kaggle/working/two_phase_multiseed")
RUNS_DIR  = OUT_DIR / "runs"
CACHE_DIR = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/kaggle/working")
for d in (OUT_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PyTorch     :", torch.__version__)
print("TorchVision :", torchvision.__version__)
print("Device      :", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "NO GPU")
print("Runs planned:", len(RUN_MODELS) * len(SEEDS), "=", len(RUN_MODELS), "models x", len(SEEDS), "seeds")

# ============================================================
# 01b — RUNS_DIR OVERRIDE: point to the already-trained checkpoints
# (this session does no training; checkpoints come from the earlier
# committed notebook, mounted as a Kaggle input)
# ============================================================
RUNS_DIR = Path("/kaggle/input/notebooks/kbmhkhan/2-sheed-multiple-phase/two_phase_multiseed/runs")
_ckpt_found = len(list(RUNS_DIR.glob("*_best.pt")))
print("RUNS_DIR override ->", RUNS_DIR, "| checkpoints found:", _ckpt_found, "/", len(RUN_MODELS) * len(SEEDS))
assert _ckpt_found == len(RUN_MODELS) * len(SEEDS), (
    "Checkpoint count mismatch -- check that the '2-Sheed-multiple_phase' notebook "
    "output is attached as an Input to this notebook before continuing."
)

# ============================================================
# 02 — TORCH-FREE HELPERS (metrics, stats, N=624 matcher)
# ============================================================
def softmax_np(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def ece_score(probs, y, n_bins=15):
    conf = probs.max(1); pred = probs.argmax(1); correct = (pred == y).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def nll_score(probs, y):
    return float(-np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1.0)).mean())

def fit_temperature(logits, y):
    from scipy.optimize import minimize_scalar
    f = lambda logT: nll_score(softmax_np(logits, float(np.exp(logT))), y)
    r = minimize_scalar(f, bounds=(-2.5, 2.5), method="bounded")
    return float(np.exp(r.x))

def metrics_from_logits(logits, y, T=1.0):
    p = softmax_np(logits, T); pred = p.argmax(1); labels = list(range(NUM_CLASSES))
    d = {
        "acc": float((pred == y).mean()),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=labels, zero_division=0)),
        "ece": ece_score(p, y, ECE_BINS),
        "nll": nll_score(p, y),
    }
    per = f1_score(y, pred, average=None, labels=labels, zero_division=0)
    for c, v in zip(CLASS_NAMES, per):
        d[f"f1_{c}"] = float(v)
    return d

def name_matches(name, prefixes):
    return any(name == p or name.startswith(p + ".") for p in prefixes)

def is_ancestor(name, prefixes):
    return any(p.startswith(name + ".") for p in prefixes)

def _norm_key(s):
    return str(s).strip().replace("\\", "/").split("/")[-1].lower()

def identify_n624(test_df, n624_df, expected_n=624):
    test_keys = test_df["filename"].map(_norm_key)
    key_set = set(test_keys)
    best_col, best_hits = None, -1
    for c in n624_df.columns:
        if not (n624_df[c].dtype == object or pd.api.types.is_string_dtype(n624_df[c])):
            continue
        hits = int(n624_df[c].map(_norm_key).isin(key_set).sum())
        if hits > best_hits:
            best_col, best_hits = c, hits
    if best_col is None:
        raise ValueError(f"No text column in N624 list. Columns: {list(n624_df.columns)}")
    wanted = set(n624_df[best_col].map(_norm_key))
    sub = test_df[test_keys.isin(wanted)].copy()
    if len(sub) != expected_n:
        raise ValueError(
            f"Matched {len(sub)} test rows (expected {expected_n}) using column '{best_col}' "
            f"({best_hits} hits). Columns available: {list(n624_df.columns)}. Set N624_COLUMN manually."
        )
    return sub, best_col
print("✓ helpers ready")

# ============================================================
# 03 — LOCKED DATASET + SPLITS + N=624 LEAKAGE-FREE TEST SET
# ============================================================
DATASET_ROOT = None
for p in KAGGLE_INPUT.rglob("Final_Waste_Dataset"):
    if (p / "images").is_dir() and (p / "research_metadata").is_dir():
        DATASET_ROOT = p
        break
assert DATASET_ROOT is not None, "Final_Waste_Dataset (images + research_metadata) not found under /kaggle/input"
IMAGE_ROOT = DATASET_ROOT / "images"
META = DATASET_ROOT / "research_metadata"
print("DATASET_ROOT:", DATASET_ROOT)

final_df = pd.read_csv(META / "final_metadata.csv").reset_index(drop=True)
train_df = pd.read_csv(META / "train_final.csv")
val_df   = pd.read_csv(META / "validation_final.csv")
test_df  = pd.read_csv(META / "test_final.csv")

counts_tuple = (len(final_df), len(train_df), len(val_df), len(test_df))
assert counts_tuple == (10668, 7245, 1552, 1871), counts_tuple

tp = set(train_df.original_path)
vp = set(val_df.original_path)
sp = set(test_df.original_path)
assert not (tp & vp) and not (tp & sp) and not (vp & sp), "split leakage by path"
assert final_df.original_path.is_unique

N624_CSV_OVERRIDE = None
N624_COLUMN = None
n624_path = Path(N624_CSV_OVERRIDE) if N624_CSV_OVERRIDE else None
if n624_path is None:
    for name in ["REAL_N624_PER_SAMPLE_LIST.csv", "AUTHORITATIVE_FINAL_LEAKAGE_FREE_N624.csv"]:
        hits = list(KAGGLE_INPUT.rglob(name))
        if hits:
            n624_path = hits[0]
            break
assert n624_path is not None and n624_path.exists(), (
    "N624 list not found. Attach Final_Waste_4_Model_Comparative_Research as input or set N624_CSV_OVERRIDE."
)
n624_df = pd.read_csv(n624_path)
print("N624 file:", n624_path, "| rows:", len(n624_df), "| columns:", list(n624_df.columns))

if N624_COLUMN:
    keys = set(n624_df[N624_COLUMN].map(_norm_key))
    test624_df = test_df[test_df["filename"].map(_norm_key).isin(keys)].copy()
    assert len(test624_df) == 624
else:
    test624_df, col_used = identify_n624(test_df, n624_df, 624)
    print("N624 matched on column:", col_used)

print("\nTest N=624 by source:\n", test624_df["source"].value_counts())
print("Test N=624 by class :\n", test624_df["class"].value_counts().reindex(CLASS_NAMES))

for df in (train_df, val_df, test624_df):
    df["label"] = df["class"].map(CLASS_TO_IDX)
    assert df["label"].notna().all()
    df["label"] = df["label"].astype(int)

counts = np.bincount(train_df["label"].values, minlength=NUM_CLASSES).astype(np.float64)
CLASS_WEIGHTS = torch.tensor(len(train_df) / (NUM_CLASSES * counts), dtype=torch.float32, device=DEVICE)
print("\nTrain class counts:", dict(zip(CLASS_NAMES, counts.astype(int))))

# ============================================================
# 04 — ONE-TIME RESIZED IMAGE CACHE (224x224 uint8)
# ============================================================
IMG_INDEX = {c: {p.name.lower(): p for p in (IMAGE_ROOT / c).iterdir()} for c in CLASS_NAMES}
CACHE_FILE = CACHE_DIR / f"waste_cache_{IMG_SIZE}.npy"
CACHE_META = CACHE_DIR / f"waste_cache_{IMG_SIZE}.json"
_resize = transforms.Resize((IMG_SIZE, IMG_SIZE))

def _load_resized(args):
    cls, fname = args
    path = IMG_INDEX[str(cls).strip()][str(fname).strip().lower()]
    with Image.open(path) as im:
        return np.asarray(_resize(im.convert("RGB")), dtype=np.uint8)

N_ALL = len(final_df)
cache_ok = CACHE_FILE.exists() and CACHE_META.exists() and json.loads(CACHE_META.read_text()).get("n") == N_ALL
if cache_ok:
    print("✓ cache found:", CACHE_FILE)
else:
    print(f"Building cache for {N_ALL} images ...")
    t0 = time.time()
    arr = np.lib.format.open_memmap(CACHE_FILE, mode="w+", dtype=np.uint8, shape=(N_ALL, IMG_SIZE, IMG_SIZE, 3))
    jobs = list(zip(final_df["class"], final_df["filename"]))
    with ThreadPoolExecutor(max_workers=4) as ex:
        for i, a in enumerate(ex.map(_load_resized, jobs)):
            arr[i] = a
            if (i + 1) % 1000 == 0:
                print(f"  {i+1}/{N_ALL}  ({time.time()-t0:.0f}s)")
    arr.flush(); del arr
    CACHE_META.write_text(json.dumps({"n": N_ALL, "img_size": IMG_SIZE}))
    print(f"✓ cache built in {time.time()-t0:.0f}s")

CACHE = np.load(CACHE_FILE, mmap_mode="r")
row_of = {p: i for i, p in enumerate(final_df["original_path"])}
for df in (train_df, val_df, test624_df):
    df["cache_idx"] = df["original_path"].map(row_of)
    assert df["cache_idx"].notna().all()
    df["cache_idx"] = df["cache_idx"].astype(int)
print("Cache shape:", CACHE.shape)

# ============================================================
# 05 — TRANSFORMS + DATALOADERS
# ============================================================
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class CachedDataset(Dataset):
    def __init__(self, df, tf):
        self.idx = df["cache_idx"].values
        self.y = df["label"].values
        self.tf = tf
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        img = Image.fromarray(np.array(CACHE[self.idx[i]]))
        return self.tf(img), int(self.y[i])

_src = test624_df["source"].astype(str).str.strip().str.lower().values
BD_MASK, INTL_MASK = (_src == "bangladesh"), (_src == "international")
print(f"train {len(train_df)} | test(N=624) {len(test624_df)}  [BD {BD_MASK.sum()}, INTL {INTL_MASK.sum()}]")
assert len(test624_df) == 624 and int(BD_MASK.sum()) == 582 and int(INTL_MASK.sum()) == 42

# ============================================================
# 06 — MODEL FACTORY
# ============================================================
def build_model(name):
    if name == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "densenet121":
        m = models.densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, NUM_CLASSES)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    elif name == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    return m

print("\n" + "=" * 60)
print("SETUP COMPLETE -- test624_df, BD_MASK, build_model all ready.")
print("=" * 60)


# ============================================================
# CDFG v3 — ROOT-CAUSE FIX for the p0-blowup, full re-run (all 4 models)
# ============================================================
# What was wrong: deletion_auc_tensor normalised the deletion curve by
# dividing every probability by probs[0] (the undeleted image's
# confidence). When probs[0] was itself near-zero (a handful of
# low-confidence-but-correct images), the ratio exploded to values like
# 76,000 instead of staying near [0, a few]. This dominated the mean
# and, for DenseNet121 and MobileNetV3 specifically, flipped the sign
# of the BD-vs-INTL gap.
#
# Fix: clip the ratio to a physically sane bound (ratio_cap=20) instead
# of leaving it unbounded. Well-behaved images (the vast majority) are
# completely unaffected -- verified on synthetic cases before this run.
#
# This writes to a NEW directory (cdfg_v3/) so nothing overwrites your
# existing backed-up results. All 4 models, all 5 seeds -- resumable,
# same memory-safe mini-batching as before.
# ============================================================
import gc, re
import numpy as np
import pandas as pd
import torch

CDFG_V3_DIR = OUT_DIR / "cdfg_v3"
CDFG_V3_DIR.mkdir(parents=True, exist_ok=True)
CDFG_STEPS = 10
CDFG_RANDOM_REPEATS = 3
RATIO_CAP = 20.0   # physically sane upper bound on probs/p0; see test above

# ---------------- preflight GPU check (same as before) ----------------
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    reserv = torch.cuda.memory_reserved(0) / 1024**3
    print(f"GPU check -> allocated {alloc:.3f} GB | reserved {reserv:.3f} GB")
    if alloc > 0.5 or reserv > 0.5:
        raise RuntimeError(
            f"GPU is NOT clean (allocated={alloc:.2f} GB). Restart Session first "
            "(Kaggle menu, NOT Factory Reset), re-run cells 1,3,4,5,6,7,8, THEN this cell."
        )
    print("✓ GPU is clean, safe to proceed.")


def safe_load_state(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    return checkpoint


def target_layer_for(model, model_name):
    if model_name == "resnet50":
        return model.layer4[-1]
    elif model_name == "densenet121":
        return model.features.denseblock4
    elif model_name in ("efficientnet_b0", "mobilenet_v3_large"):
        return model.features[-1]
    raise ValueError(model_name)


def gradcam_map(model, x, class_idx, layer):
    captured = {}

    def hook(_m, _in, out):
        captured["activation"] = out
        out.register_hook(lambda grad: captured.__setitem__("gradient", grad))

    handle = layer.register_forward_hook(hook)
    try:
        model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = model(x.requires_grad_(True))
            logits[0, class_idx].backward()
        acts, grads = captured["activation"], captured["gradient"]
        cam = torch.relu((grads.mean((2, 3), keepdim=True) * acts).sum(1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        result = cam.detach()[None, ...]
    finally:
        handle.remove()
        model.zero_grad(set_to_none=True)
    del captured
    return result


def deletion_auc_tensor(model, x, class_idx, order, steps=CDFG_STEPS, sub_batch=4, ratio_cap=RATIO_CAP):
    h, w = x.shape[-2:]
    flat_order = order.reshape(-1)
    variants = []
    for step in range(steps + 1):
        n_remove = round(step * h * w / steps)
        masked = x.clone()
        if n_remove:
            idx = flat_order[:n_remove]
            yy, xx = torch.div(idx, w, rounding_mode="floor"), idx % w
            masked[:, :, yy, xx] = 0.0
        variants.append(masked)
    probs_chunks = []
    with torch.no_grad():
        for i in range(0, len(variants), sub_batch):
            batch = torch.cat(variants[i:i + sub_batch], dim=0)
            logits = model(batch)
            p = torch.softmax(logits, dim=1)[:, class_idx].float().cpu().numpy()
            probs_chunks.append(p)
            del batch, logits
    del variants
    probs = np.concatenate(probs_chunks)
    p0 = max(float(probs[0]), 1e-8)
    ratio = np.clip(probs / p0, 0.0, ratio_cap)   # <-- THE FIX
    return float(np.sum((ratio[:-1] + ratio[1:]) * 0.5) / steps)


def gpu_mem_gb():
    if not torch.cuda.is_available():
        return 0.0, 0.0
    return (torch.cuda.memory_allocated(0) / 1024**3, torch.cuda.memory_reserved(0) / 1024**3)


# ---------------- resume: skip (model,seed) already done in cdfg_v3/ ----------------
done = set()
for f in CDFG_V3_DIR.glob("*_per_image.csv"):
    m = re.match(r"(.+)_seed(\d+)_per_image\.csv$", f.name)
    if m:
        done.add((m.group(1), int(m.group(2))))
print(f"Resuming v3: {len(done)} (model,seed) runs already done -> {sorted(done)}")

for model_name in RUN_MODELS:
    for seed in SEEDS:
        if (model_name, seed) in done:
            print(f"SKIP (already done): {model_name} seed {seed}")
            continue
        ckpt_path = RUNS_DIR / f"{model_name}_seed{seed}_best.pt"
        if not ckpt_path.exists():
            print("SKIP: missing checkpoint", ckpt_path); continue

        model = build_model(model_name).to(DEVICE).eval()
        model.load_state_dict(safe_load_state(ckpt_path))
        layer = target_layer_for(model, model_name)

        rows = []
        for index in range(len(test624_df)):
            row = test624_df.iloc[index]
            x, y = CachedDataset(test624_df.iloc[[index]], eval_tf)[0]
            x = x.unsqueeze(0).to(DEVICE); y = int(y)
            cam = gradcam_map(model, x, y, layer)[0, 0]
            cam_order = torch.argsort(cam.flatten(), descending=True, stable=True)
            auc_cam = deletion_auc_tensor(model, x, y, cam_order)
            rnd_aucs = []
            img_seed = int(seed + index * 1009)
            for rep in range(CDFG_RANDOM_REPEATS):
                gen = torch.Generator(device="cpu").manual_seed(img_seed + rep)
                rnd_order = torch.randperm(IMG_SIZE * IMG_SIZE, generator=gen).to(DEVICE)
                rnd_aucs.append(deletion_auc_tensor(model, x, y, rnd_order))
            with torch.no_grad():
                pred_correct = int(model(x).argmax(1).item() == y)
            rows.append(dict(model=model_name, seed=seed, image=row["filename"], source=row["source"],
                              label=y, correct=pred_correct,
                              auc_cam=auc_cam, auc_random=float(np.mean(rnd_aucs)),
                              faithfulness=float(np.mean(rnd_aucs) - auc_cam)))
            del x, cam, cam_order
            if index % 150 == 0:
                a, r = gpu_mem_gb()
                print(f"  {model_name} seed={seed} img {index}/{len(test624_df)}  [gpu alloc={a:.2f}GB]")

        run_df = pd.DataFrame(rows)
        run_df.to_csv(CDFG_V3_DIR / f"{model_name}_seed{seed}_per_image.csv", index=False)
        print(f"✓ done {model_name} seed {seed}: n={len(rows)}  "
              f"max|faithfulness|={run_df['faithfulness'].abs().max():.3f}  (should be well under {RATIO_CAP})")
        del model, rows, run_df
        gc.collect(); torch.cuda.empty_cache()

# ---------------- merge + final stats ----------------
all_csvs = sorted(CDFG_V3_DIR.glob("*_per_image.csv"))
if len(all_csvs) < len(RUN_MODELS) * len(SEEDS):
    print(f"\nOnly {len(all_csvs)}/{len(RUN_MODELS)*len(SEEDS)} runs done -- re-run this SAME cell to resume.")
else:
    full = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
    full["domain"] = full["source"].astype(str).str.strip().str.lower()
    full.to_csv(CDFG_V3_DIR / "cdfg_all_per_image_FIXED.csv", index=False)
    print(f"\nAll {len(all_csvs)} runs complete. max|faithfulness| overall = {full['faithfulness'].abs().max():.3f}")

    from scipy import stats as sstats

    def stratified_gap(v_a, c_a, v_b, c_b):
        gaps = [v_a[c_a == c].mean() - v_b[c_b == c].mean()
                for c in np.intersect1d(np.unique(c_a), np.unique(c_b))]
        return float(np.mean(gaps)) if gaps else float("nan")

    def gap_with_stats(v_a, c_a, v_b, c_b, n_boot=3000, n_perm=3000, seed=0):
        rng = np.random.default_rng(seed)
        v_a, c_a, v_b, c_b = map(np.asarray, (v_a, c_a, v_b, c_b))
        obs = stratified_gap(v_a, c_a, v_b, c_b)
        idx_a = {c: np.where(c_a == c)[0] for c in np.unique(c_a)}
        idx_b = {c: np.where(c_b == c)[0] for c in np.unique(c_b)}
        shared = [c for c in idx_a if c in idx_b]
        boots = np.empty(n_boot)
        for i in range(n_boot):
            g = [v_a[rng.choice(idx_a[c], len(idx_a[c]))].mean() - v_b[rng.choice(idx_b[c], len(idx_b[c]))].mean()
                 for c in shared]
            boots[i] = np.mean(g)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        ge = 0
        for _ in range(n_perm):
            g = []
            for c in shared:
                pool = np.concatenate([v_a[idx_a[c]], v_b[idx_b[c]]])
                rng.shuffle(pool)
                na = len(idx_a[c])
                g.append(pool[:na].mean() - pool[na:].mean())
            if abs(np.mean(g)) >= abs(obs):
                ge += 1
        p = (ge + 1) / (n_perm + 1)
        return obs, (float(lo), float(hi)), p

    print("\n=== FINAL (root-cause fixed) image-level bootstrap, INTL - BD ===")
    summary_rows = []
    for m, g in full.groupby("model"):
        bd = g[g.domain == "bangladesh"]
        it = g[g.domain == "international"]
        obs, ci, p = gap_with_stats(it["faithfulness"].values, it["label"].values,
                                     bd["faithfulness"].values, bd["label"].values)
        sig = "reliable" if ci[0] * ci[1] > 0 else "not reliable (CI crosses 0)"
        print(f"{m:22s} gap(INTL-BD)={obs:+.4f}  95% CI=[{ci[0]:+.4f}, {ci[1]:+.4f}]  perm p={p:.4f}  {sig}")
        summary_rows.append(dict(model=m, gap_INTL_minus_BD=obs, ci_lo=ci[0], ci_hi=ci[1], perm_p=p))

    pd.DataFrame(summary_rows).to_csv(CDFG_V3_DIR / "FINAL_gap_summary_FIXED.csv", index=False)
    print("\nSaved: FINAL_gap_summary_FIXED.csv -- these are the numbers to put in the thesis.")

PyTorch     : 2.11.0+cu128
TorchVision : 0.26.0+cu128
Device      : cuda Tesla T4
Runs planned: 20 = 4 models x 5 seeds
RUNS_DIR override -> /kaggle/input/notebooks/kbmhkhan/2-sheed-multiple-phase/two_phase_multiseed/runs | checkpoints found: 20 / 20
✓ helpers ready
DATASET_ROOT: /kaggle/input/notebooks/kbmhkhan/full-final-waste-dataset-structure/Final_Waste_Dataset
N624 file: /kaggle/input/code/deleted/132451660/four_model_comparison/REAL_N624_PER_SAMPLE_LIST.csv | rows: 624 | columns: ['unique_key', 'filename', 'original_path', 'class', 'true_idx', 'country', 'source']
N624 matched on column: filename

Test N=624 by source:
 source
Bangladesh       582
International     42
Name: count, dtype: int64
Test N=624 by class :
 class
Cardboard     33
Glass        128
Metal         82
Paper        199
Plastic      182
Name: count, dtype: int64

Train class counts: {'Cardboard': np.int64(1193), 'Glass': np.int64(1433), 'Metal': np.int64(1594), 'Paper': np.int64(1793), 'Plastic': np.int64(1232

In [2]:
done = sorted((OUT_DIR / "cdfg_v3").glob("*_per_image.csv"))
print(f"{len(done)} / 20 runs actually saved to disk:")
for f in done:
    print(" ", f.name)

20 / 20 runs actually saved to disk:
  densenet121_seed123_per_image.csv
  densenet121_seed2024_per_image.csv
  densenet121_seed3407_per_image.csv
  densenet121_seed42_per_image.csv
  densenet121_seed777_per_image.csv
  efficientnet_b0_seed123_per_image.csv
  efficientnet_b0_seed2024_per_image.csv
  efficientnet_b0_seed3407_per_image.csv
  efficientnet_b0_seed42_per_image.csv
  efficientnet_b0_seed777_per_image.csv
  mobilenet_v3_large_seed123_per_image.csv
  mobilenet_v3_large_seed2024_per_image.csv
  mobilenet_v3_large_seed3407_per_image.csv
  mobilenet_v3_large_seed42_per_image.csv
  mobilenet_v3_large_seed777_per_image.csv
  resnet50_seed123_per_image.csv
  resnet50_seed2024_per_image.csv
  resnet50_seed3407_per_image.csv
  resnet50_seed42_per_image.csv
  resnet50_seed777_per_image.csv


In [3]:
# ============================================================
# CDFG v3 — MERGE + FINAL STATS ONLY (no GPU, no re-training)
# Reads the 20 already-saved per-image CSVs and produces the final,
# root-cause-fixed bootstrap summary.
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as sstats

# Point this at wherever the 20 *_per_image.csv files from the v3 run
# actually are. If you're continuing in the SAME session that produced
# them, OUT_DIR is already defined and this default is correct.
try:
    CDFG_V3_DIR = OUT_DIR / "cdfg_v3"
except NameError:
    # fresh session fallback -- edit this path if needed
    CDFG_V3_DIR = Path("/kaggle/working/two_phase_multiseed/cdfg_v3")

all_csvs = sorted(CDFG_V3_DIR.glob("*_per_image.csv"))
print(f"Found {len(all_csvs)} per-image CSVs in {CDFG_V3_DIR}")
for f in all_csvs:
    print(" ", f.name)

RUN_MODELS_EXPECTED = ["mobilenet_v3_large", "resnet50", "densenet121", "efficientnet_b0"]
SEEDS_EXPECTED = [42, 123, 2024, 3407, 777]
assert len(all_csvs) == len(RUN_MODELS_EXPECTED) * len(SEEDS_EXPECTED), (
    f"Expected {len(RUN_MODELS_EXPECTED) * len(SEEDS_EXPECTED)} files, found {len(all_csvs)}. "
    "Re-run the full ALL_IN_ONE cell first -- it will resume and finish the missing runs."
)

full = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
full["domain"] = full["source"].astype(str).str.strip().str.lower()
full.to_csv(CDFG_V3_DIR / "cdfg_all_per_image_FIXED.csv", index=False)

print(f"\nTotal rows: {len(full)} (expect {624 * 20} = 12480)")
print(f"max|faithfulness| overall = {full['faithfulness'].abs().max():.3f}  (should be well under 20)")
print(f"\nPer-model max|faithfulness|:")
print(full.groupby("model")["faithfulness"].apply(lambda s: s.abs().max()))


def stratified_gap(v_a, c_a, v_b, c_b):
    gaps = [v_a[c_a == c].mean() - v_b[c_b == c].mean()
            for c in np.intersect1d(np.unique(c_a), np.unique(c_b))]
    return float(np.mean(gaps)) if gaps else float("nan")


def gap_with_stats(v_a, c_a, v_b, c_b, n_boot=3000, n_perm=3000, seed=0):
    rng = np.random.default_rng(seed)
    v_a, c_a, v_b, c_b = map(np.asarray, (v_a, c_a, v_b, c_b))
    obs = stratified_gap(v_a, c_a, v_b, c_b)
    idx_a = {c: np.where(c_a == c)[0] for c in np.unique(c_a)}
    idx_b = {c: np.where(c_b == c)[0] for c in np.unique(c_b)}
    shared = [c for c in idx_a if c in idx_b]
    boots = np.empty(n_boot)
    for i in range(n_boot):
        g = [v_a[rng.choice(idx_a[c], len(idx_a[c]))].mean() - v_b[rng.choice(idx_b[c], len(idx_b[c]))].mean()
             for c in shared]
        boots[i] = np.mean(g)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    ge = 0
    for _ in range(n_perm):
        g = []
        for c in shared:
            pool = np.concatenate([v_a[idx_a[c]], v_b[idx_b[c]]])
            rng.shuffle(pool)
            na = len(idx_a[c])
            g.append(pool[:na].mean() - pool[na:].mean())
        if abs(np.mean(g)) >= abs(obs):
            ge += 1
    p = (ge + 1) / (n_perm + 1)
    return obs, (float(lo), float(hi)), p


print("\n=== FINAL (root-cause fixed) image-level bootstrap, INTL - BD ===")
summary_rows = []
for m, g in full.groupby("model"):
    bd = g[g.domain == "bangladesh"]
    it = g[g.domain == "international"]
    obs, ci, p = gap_with_stats(it["faithfulness"].values, it["label"].values,
                                 bd["faithfulness"].values, bd["label"].values)
    sig = "reliable" if ci[0] * ci[1] > 0 else "not reliable (CI crosses 0)"
    print(f"{m:22s} gap(INTL-BD)={obs:+.4f}  95% CI=[{ci[0]:+.4f}, {ci[1]:+.4f}]  perm p={p:.4f}  {sig}")
    summary_rows.append(dict(model=m, gap_INTL_minus_BD=obs, ci_lo=ci[0], ci_hi=ci[1], perm_p=p))

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(CDFG_V3_DIR / "FINAL_gap_summary_FIXED.csv", index=False)
print("\nSaved:", CDFG_V3_DIR / "FINAL_gap_summary_FIXED.csv")

# bonus: BD/INTL accuracy for context (same domain split, from this same data)
print("\n=== BD vs INTL accuracy (for context) ===")
acc = full.groupby(["model", "domain"])["correct"].mean().unstack()
print(acc.round(4))

Found 20 per-image CSVs in /kaggle/working/two_phase_multiseed/cdfg_v3
  densenet121_seed123_per_image.csv
  densenet121_seed2024_per_image.csv
  densenet121_seed3407_per_image.csv
  densenet121_seed42_per_image.csv
  densenet121_seed777_per_image.csv
  efficientnet_b0_seed123_per_image.csv
  efficientnet_b0_seed2024_per_image.csv
  efficientnet_b0_seed3407_per_image.csv
  efficientnet_b0_seed42_per_image.csv
  efficientnet_b0_seed777_per_image.csv
  mobilenet_v3_large_seed123_per_image.csv
  mobilenet_v3_large_seed2024_per_image.csv
  mobilenet_v3_large_seed3407_per_image.csv
  mobilenet_v3_large_seed42_per_image.csv
  mobilenet_v3_large_seed777_per_image.csv
  resnet50_seed123_per_image.csv
  resnet50_seed2024_per_image.csv
  resnet50_seed3407_per_image.csv
  resnet50_seed42_per_image.csv
  resnet50_seed777_per_image.csv

Total rows: 12480 (expect 12480 = 12480)
max|faithfulness| overall = 18.688  (should be well under 20)

Per-model max|faithfulness|:
model
densenet121            7.

In [4]:
from pathlib import Path

candidates = list(Path("/kaggle/input").rglob("sam_vit_b_01ec64.pth"))

print("Found checkpoints:", len(candidates))
for p in candidates:
    print(p)

Found checkpoints: 0


In [5]:
import numpy as np
import pandas as pd

# `full` হলো cdfg_all_per_image_FIXED.csv থেকে load করা merged dataframe.
# প্রত্যাশিত columns: model, image, source, label, seed, faithfulness
df = full.copy()
df["domain"] = df["source"].astype(str).str.strip().str.lower()
df = df[df["domain"].isin(["bangladesh", "international"])].copy()

# একই 이미지র 5 seed-কে 5টি স্বাধীন ছবি হিসেবে গণনা করব না:
# প্রতি ছবির seed-গড় score-ই হবে observation.
img = (
    df.groupby(["model", "domain", "image", "label"], as_index=False)
      ["faithfulness"].mean()
)

def stratified_gap(intl, bd):
    shared = sorted(set(intl) & set(bd))
    return float(np.mean([intl[c].mean() - bd[c].mean() for c in shared]))

def image_level_tests(g, n_boot=10000, n_perm=10000, seed=20261012):
    rng = np.random.default_rng(seed)
    intl = {
        c: x["faithfulness"].to_numpy()
        for c, x in g[g["domain"] == "international"].groupby("label")
    }
    bd = {
        c: x["faithfulness"].to_numpy()
        for c, x in g[g["domain"] == "bangladesh"].groupby("label")
    }
    classes = sorted(set(intl) & set(bd))
    if not classes:
        raise ValueError("No classes shared by both domains.")

    observed = stratified_gap(intl, bd)

    # Class-stratified bootstrap: resample unique images within each domain/class.
    boots = np.empty(n_boot)
    for b in range(n_boot):
        gaps = []
        for c in classes:
            a = rng.choice(intl[c], size=len(intl[c]), replace=True)
            d = rng.choice(bd[c], size=len(bd[c]), replace=True)
            gaps.append(a.mean() - d.mean())
        boots[b] = np.mean(gaps)
    lo, hi = np.percentile(boots, [2.5, 97.5])

    # Class-stratified permutation on image-level observations.
    extreme = 0
    for _ in range(n_perm):
        gaps = []
        for c in classes:
            pool = np.concatenate([intl[c], bd[c]])
            shuffled = rng.permutation(pool)
            n_intl = len(intl[c])
            gaps.append(shuffled[:n_intl].mean() - shuffled[n_intl:].mean())
        if abs(np.mean(gaps)) >= abs(observed):
            extreme += 1

    p = (extreme + 1) / (n_perm + 1)
    return observed, float(lo), float(hi), float(p)

rows = []
for model, g in img.groupby("model"):
    gap, lo, hi, p = image_level_tests(g)
    rows.append({
        "model": model,
        "gap_INTL_minus_BD": gap,
        "ci_lo": lo,
        "ci_hi": hi,
        "perm_p": p,
        "n_unique_images": g["image"].nunique(),
    })

result = pd.DataFrame(rows)

# Holm correction across the four model-level permutation tests.
order = np.argsort(result["perm_p"].to_numpy())
pvals = result["perm_p"].to_numpy()
adjusted = np.empty(len(pvals))
running_max = 0.0
m = len(pvals)
for rank, idx in enumerate(order):
    running_max = max(running_max, (m - rank) * pvals[idx])
    adjusted[idx] = min(running_max, 1.0)

result["perm_p_holm"] = adjusted
result["decision"] = np.where(
    (result["ci_lo"] * result["ci_hi"] > 0) & (result["perm_p_holm"] < 0.05),
    "significant by both",
    "not confirmed",
)

display(result.sort_values("model"))

,model,gap_INTL_minus_BD,ci_lo,ci_hi,perm_p,n_unique_images,perm_p_holm,decision
0,densenet121,0.205890,-0.075565,0.515241,0.018198,624,0.072793,not confirmed
1,efficientnet_b0,-0.230253,-0.536923,-0.000885,0.292971,624,0.585941,not confirmed
2,mobilenet_v3_large,0.028870,-0.424138,0.540724,0.964404,624,0.964404,not confirmed
3,resnet50,-0.168449,-0.300521,-0.046233,0.154485,624,0.463454,not confirmed


In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

# FIX: `full` was only in a previous kernel's memory. Load it fresh from
# the CSV that the merge cell already saved to disk.
try:
    CDFG_V3_DIR = OUT_DIR / "cdfg_v3"
except NameError:
    CDFG_V3_DIR = Path("/kaggle/working/two_phase_multiseed/cdfg_v3")

full = pd.read_csv(CDFG_V3_DIR / "cdfg_all_per_image_FIXED.csv")

# `full` হলো cdfg_all_per_image_FIXED.csv থেকে load করা merged dataframe।
# প্রত্যাশিত columns: model, image, source, label, seed, faithfulness
df = full.copy()
df["domain"] = df["source"].astype(str).str.strip().str.lower()
df = df[df["domain"].isin(["bangladesh", "international"])].copy()

# একই ছবির 5 seed-কে 5টি স্বাধীন ছবি হিসেবে গণনা করব না:
# প্রতি ছবির seed-গড় score-ই হবে observation।
img = (
    df.groupby(["model", "domain", "image", "label"], as_index=False)
      ["faithfulness"].mean()
)

def stratified_gap(intl, bd):
    shared = sorted(set(intl) & set(bd))
    return float(np.mean([intl[c].mean() - bd[c].mean() for c in shared]))

def image_level_tests(g, n_boot=10000, n_perm=10000, seed=20261012):
    rng = np.random.default_rng(seed)
    intl = {
        c: x["faithfulness"].to_numpy()
        for c, x in g[g["domain"] == "international"].groupby("label")
    }
    bd = {
        c: x["faithfulness"].to_numpy()
        for c, x in g[g["domain"] == "bangladesh"].groupby("label")
    }
    classes = sorted(set(intl) & set(bd))
    if not classes:
        raise ValueError("No classes shared by both domains.")

    observed = stratified_gap(intl, bd)

    boots = np.empty(n_boot)
    for b in range(n_boot):
        gaps = []
        for c in classes:
            a = rng.choice(intl[c], size=len(intl[c]), replace=True)
            d = rng.choice(bd[c], size=len(bd[c]), replace=True)
            gaps.append(a.mean() - d.mean())
        boots[b] = np.mean(gaps)
    lo, hi = np.percentile(boots, [2.5, 97.5])

    extreme = 0
    for _ in range(n_perm):
        gaps = []
        for c in classes:
            pool = np.concatenate([intl[c], bd[c]])
            shuffled = rng.permutation(pool)
            n_intl = len(intl[c])
            gaps.append(shuffled[:n_intl].mean() - shuffled[n_intl:].mean())
        if abs(np.mean(gaps)) >= abs(observed):
            extreme += 1

    p = (extreme + 1) / (n_perm + 1)
    return observed, float(lo), float(hi), float(p)

rows = []
for model, g in img.groupby("model"):
    gap, lo, hi, p = image_level_tests(g)
    rows.append({
        "model": model,
        "gap_INTL_minus_BD": gap,
        "ci_lo": lo,
        "ci_hi": hi,
        "perm_p": p,
        "n_unique_images": g["image"].nunique(),
    })

result = pd.DataFrame(rows)

order = np.argsort(result["perm_p"].to_numpy())
pvals = result["perm_p"].to_numpy()
adjusted = np.empty(len(pvals))
running_max = 0.0
m = len(pvals)
for rank, idx in enumerate(order):
    running_max = max(running_max, (m - rank) * pvals[idx])
    adjusted[idx] = min(running_max, 1.0)

result["perm_p_holm"] = adjusted
result["decision"] = np.where(
    (result["ci_lo"] * result["ci_hi"] > 0) & (result["perm_p_holm"] < 0.05),
    "significant by both",
    "not confirmed",
)

result.to_csv(CDFG_V3_DIR / "FINAL_gap_summary_IMAGE_LEVEL_HOLM.csv", index=False)
print(result.sort_values("model").to_string(index=False))
print("\nSaved:", CDFG_V3_DIR / "FINAL_gap_summary_IMAGE_LEVEL_HOLM.csv")
print("\nThis is the MORE RIGOROUS result (per-image averaging across seeds")
print("avoids pseudo-replication; Holm correction controls for testing 4 models).")
print("Use THIS table in the thesis, not the naive per-(image,seed) version.")

             model  gap_INTL_minus_BD     ci_lo     ci_hi   perm_p  n_unique_images  perm_p_holm      decision
       densenet121           0.205890 -0.075565  0.515241 0.018198              624     0.072793 not confirmed
   efficientnet_b0          -0.230253 -0.536923 -0.000885 0.292971              624     0.585941 not confirmed
mobilenet_v3_large           0.028870 -0.424138  0.540724 0.964404              624     0.964404 not confirmed
          resnet50          -0.168449 -0.300521 -0.046233 0.154485              624     0.463454 not confirmed

Saved: /kaggle/working/two_phase_multiseed/cdfg_v3/FINAL_gap_summary_IMAGE_LEVEL_HOLM.csv

This is the MORE RIGOROUS result (per-image averaging across seeds
avoids pseudo-replication; Holm correction controls for testing 4 models).
Use THIS table in the thesis, not the naive per-(image,seed) version.
